# Loan Default Prediction System

Build, compare and evaluate Logistic Regression, Random Forest, XGBoost and SVM.

Dataset: German Credit Data — 1,000 real loan applicants, sourced from the UCI Statlog German Credit dataset.

In [ ]:
!pip -q install xgboost


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
import joblib


In [ ]:
# Load real dataset
url = "https://raw.githubusercontent.com/srinibas-masanta/Credit-Risk-Modeling/main/german_credit_data.csv"
df = pd.read_csv(url)

df.head()


In [ ]:
# Basic checks
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
        
print("\nMissing values:")
print(df.isnull().sum())


In [ ]:
# Target distribution
print(df["Risk"].value_counts())

df["Risk"].value_counts().plot(kind="bar")
plt.xlabel("Risk")
plt.ylabel("Count")
plt.title("Credit Risk Distribution")
plt.show()


In [ ]:
# Simple EDA
print(df.groupby("Risk")[["Age", "Credit amount", "Duration"]].mean().round(2))


In [ ]:
# Convert target to 0 and 1
df["Default"] = (df["Risk"].str.lower() == "bad").astype(int)

df = df.drop(columns=["Risk"])
df = df.drop(columns=[df.columns[0]])

df.head()


In [ ]:
# Save project dataset
df.to_csv("loan_default_dataset.csv", index=False)
        
print("Saved: loan_default_dataset.csv")


In [ ]:
# Separate features and target
X = df.drop(columns=["Default"])
y = df["Default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


In [ ]:
# Preprocessing
numeric_features = ["Age", "Credit amount", "Duration"]
categorical_features = [
    "Sex", "Job", "Housing", "Saving accounts",
    "Checking account", "Purpose"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


In [ ]:
# Define models
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()
scale_pos_weight = negative / positive

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    ),
    "SVM": SVC(
        C=1.0,
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=42
    )
}


In [ ]:
# Build pipelines
pipelines = {
    name: Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}


In [ ]:
# Train and evaluate
results = []
predictions = {}
probabilities = {}

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    predictions[name] = y_pred
    probabilities[name] = y_prob

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
results_df.round(3)


In [ ]:
# Cross-validation ROC-AUC
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, pipeline in pipelines.items():
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )
    cv_results.append({
        "Model": name,
        "CV ROC-AUC": scores.mean(),
        "CV Std": scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values("CV ROC-AUC", ascending=False)
cv_df.round(3)


In [ ]:
# Compare test ROC-AUC
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["ROC-AUC"])
plt.ylabel("ROC-AUC")
plt.xlabel("Model")
plt.title("Model Comparison")
plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.show()


In [ ]:
# ROC curves
plt.figure(figsize=(8, 6))

for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()


In [ ]:
# Select best model by test ROC-AUC
best_model_name = results_df.iloc[0]["Model"]
best_model = pipelines[best_model_name]

print("Best model:", best_model_name)


In [ ]:
# Classification report
print(classification_report(
    y_test,
    predictions[best_model_name],
    target_names=["No Default", "Default"]
))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, predictions[best_model_name])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Default", "Default"]
)
disp.plot()
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()


In [ ]:
# Feature importance using permutation importance
importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": importance.importances_mean
}).sort_values("Importance", ascending=False)

print(importance_df.round(4))


In [ ]:
# Plot feature importance
top_features = importance_df.head(10).sort_values("Importance")

plt.figure(figsize=(9, 5))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Mean decrease in ROC-AUC")
plt.ylabel("Feature")
plt.title(f"Top Features - {best_model_name}")
plt.show()


In [ ]:
# Save results
results_df.to_csv("model_comparison.csv", index=False)
importance_df.to_csv("feature_importance.csv", index=False)
joblib.dump(best_model, "best_loan_default_model.pkl")

print("Saved:")
print("- model_comparison.csv")
print("- feature_importance.csv")
print("- best_loan_default_model.pkl")


In [ ]:
# Example prediction
new_applicant = pd.DataFrame([{
    "Age": 30,
    "Sex": "male",
    "Job": 2,
    "Housing": "rent",
    "Saving accounts": "little",
    "Checking account": "little",
    "Credit amount": 3000,
    "Duration": 18,
    "Purpose": "car"
}])

prediction = best_model.predict(new_applicant)[0]
probability = best_model.predict_proba(new_applicant)[0, 1]

print("Prediction:", "Default" if prediction == 1 else "No Default")
print("Default probability:", round(probability, 3))


## Final conclusion

Compare the four classifiers using ROC-AUC, F1, precision and recall.
Select the best model based on the evaluation results, inspect the confusion matrix, and use the feature importance plot to explain the prediction.

Note: this is an educational credit-risk project. The dataset is small and historical, so the results should not be treated as a production lending model.